# 04 — Mega 500K Game Simulation

Run **500,000 AI vs AI games** across a wide range of skill levels (depth 3 through 20).

- 20 matchup pairings grouped into 6 tiers: Beginner, Intermediate, Advanced, Expert, Master, Grandmaster
- 90% of games use a random first move (for opener statistics); 10% are fully AI-chosen
- Tiered AI search budgets so deep games actually search deeply
- Checkpoint/resume: safe to interrupt and re-run
- Results stored in a separate `data/mega_games.sqlite`

**For actual runs, use the standalone script** — it has all the memory/IPC optimizations:
```bash
# Full run (resume-safe, run under tmux or nohup):
nohup python3 04_mega_simulation.py > mega_run.log 2>&1 &

# Calibrate only:
python3 04_mega_simulation.py --calibrate

# DB summary only:
python3 04_mega_simulation.py --summary
```

This notebook remains useful for the summary/analysis cells after the run completes.

### Improvement ideas for this mega run

1. **Tiered AI budgets** — depth 20 with the default 25K node cap never truly searches that deep. The grandmaster config here gives 200K nodes and 30s per move.
2. **WAL mode for SQLite** — enables concurrent reads during writes, so you can peek at results while the run is still going.
3. **Batch insert size = 500** — fewer commits, faster I/O at this volume.
4. **Schedule sorted fast-first** — beginner matchups complete in minutes, giving immediate feedback. Deep matchups run last.
5. **Board-after compression** (optional) — `board_after` JSON is the biggest storage cost. Storing it only every Nth move for deep games saves ~60% disk, at the cost of harder per-move reconstruction.
6. **Run as a detached script** — for multi-day runs, export via `jupyter nbconvert` and run under `nohup`/`tmux` (see final cell).
7. **Stochastic top-k = 1 for deterministic games** — the 10% non-random-opener batch also uses `stochastic_top_k=1` for fully deterministic play, giving a pure best-play baseline.
8. **Progress JSON checkpoint** — `data/mega_progress.json` is written after each batch for easy external monitoring.
9. **Calibration cell** — estimates total wall-clock time per tier from a quick 10-game sample before the full run.

In [1]:
import os, sys, time, json, sqlite3, random, math
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timedelta

try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

import pandas as pd

# Ensure the engine package is importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from engine.board import initial_game_state, apply_move_to_board
from engine.ai import (
    get_next_best_move, get_all_possible_moves,
    apply_move_ai, is_game_over,
)
from engine.runner import play_one_game, worker as _worker

## Configuration

In [2]:
# === MATCHUP SCHEDULE ===
# Each entry: (depth_a, depth_b, n_games, tier_label)
# depth_a always <= depth_b. Both color orderings are played (a-as-white, b-as-white).

SCHEDULE = [
    # --- Beginner tier (very fast, ~ms per game) ---
    (3,  3,   20_000, 'beginner'),
    (3,  6,   25_000, 'beginner'),
    (6,  6,   30_000, 'beginner'),
    # --- Intermediate tier ---
    (3,  9,   20_000, 'intermediate'),
    (6,  9,   35_000, 'intermediate'),
    (9,  9,   50_000, 'intermediate'),
    # --- Advanced tier (core competitive range) ---
    (3,  12,  15_000, 'advanced'),
    (6,  12,  25_000, 'advanced'),
    (9,  12,  50_000, 'advanced'),
    (12, 12,  60_000, 'advanced'),
    # --- Expert tier ---
    (9,  15,  20_000, 'expert'),
    (12, 15,  40_000, 'expert'),
    (15, 15,  25_000, 'expert'),
    # --- Master tier ---
    (12, 18,  15_000, 'master'),
    (15, 18,  15_000, 'master'),
    (18, 18,   8_000, 'master'),
    # --- Grandmaster tier (depth 20 — target >= 10K games) ---
    (12, 20,  17_000, 'grandmaster'),
    (15, 20,  10_000, 'grandmaster'),
    (18, 20,   8_000, 'grandmaster'),
    (20, 20,  12_000, 'grandmaster'),
]

TOTAL_GAMES = sum(n for _, _, n, _ in SCHEDULE)
DEPTH_20_GAMES = sum(n for a, b, n, _ in SCHEDULE if a == 20 or b == 20)

print(f'Total scheduled games: {TOTAL_GAMES:,}')
print(f'Depth-20 games:        {DEPTH_20_GAMES:,}')
print(f'Matchup pairings:      {len(SCHEDULE)}')

Total scheduled games: 500,000
Depth-20 games:        47,000
Matchup pairings:      20


In [3]:
# === TIERED AI SEARCH CONFIGS ===
# Every tier has explicit budgets. The original beginner config had
# max_ms=None / max_nodes=None which caused 6v6 to do unbounded depth-6
# searches at ~10s/game. With caps, 6v6 drops to ~77ms/game.

AI_CONFIGS = {
    'beginner': {
        'max_ms': 500,
        'max_nodes': 5_000,
        'root_probe_nodes': 50,
        'stochastic_top_k': 3,
    },
    'intermediate': {
        'max_ms': 2_000,
        'max_nodes': 15_000,
        'root_probe_nodes': 80,
        'stochastic_top_k': 3,
    },
    'advanced': {
        'max_ms': 5_000,
        'max_nodes': 25_000,
        'root_probe_nodes': 100,
        'stochastic_top_k': 3,
    },
    'expert': {
        'max_ms': 10_000,
        'max_nodes': 50_000,
        'root_probe_nodes': 150,
        'stochastic_top_k': 3,
    },
    'master': {
        'max_ms': 20_000,
        'max_nodes': 100_000,
        'root_probe_nodes': 200,
        'stochastic_top_k': 3,
    },
    'grandmaster': {
        'max_ms': 30_000,
        'max_nodes': 200_000,
        'root_probe_nodes': 300,
        'stochastic_top_k': 3,
    },
}

# For the 10% deterministic-opener games, override stochastic_top_k
# to 1 so we get fully deterministic best-play baselines.
DETERMINISTIC_OVERRIDE = {'stochastic_top_k': 1}

# === ADAPTIVE WORKER COUNTS ===
CPU_COUNT = os.cpu_count() or 4
WORKER_COUNTS = {
    'beginner':     max(1, CPU_COUNT - 1),
    'intermediate': max(1, CPU_COUNT - 1),
    'advanced':     max(1, CPU_COUNT - 1),
    'expert':       max(2, CPU_COUNT // 2),
    'master':       max(2, CPU_COUNT // 3),
    'grandmaster':  max(2, CPU_COUNT // 3),
}

# === GENERAL SETTINGS ===
MAX_MOVES = 200
BATCH_INSERT_SIZE = 500
RANDOM_FRACTION = 0.90       # 90% random first move, 10% deterministic
DB_PATH = os.path.join('data', 'mega_games.sqlite')
PROGRESS_PATH = os.path.join('data', 'mega_progress.json')

print(f'CPU count: {CPU_COUNT}')
print(f'DB path:   {DB_PATH}')
for tier, wc in WORKER_COUNTS.items():
    print(f'  {tier:15s} workers: {wc}')

CPU count: 12
DB path:   data/mega_games.sqlite
  beginner        workers: 11
  intermediate    workers: 11
  advanced        workers: 11
  expert          workers: 6
  master          workers: 4
  grandmaster     workers: 4


## Smoke Test

Run 1 game at low depth to verify the engine works before kicking off the mega run.

In [4]:
print('Running smoke test (depth 3 vs 3)...')
smoke_kwargs = {'max_ms': 10, 'max_nodes': None, 'root_probe_nodes': 50, 'stochastic_top_k': 3}
g, m = play_one_game(
    3, 3, seed=42,
    white_ai_kwargs=smoke_kwargs,
    black_ai_kwargs=smoke_kwargs,
    random_first_move=True,
)
print(f'  Winner: {g["winner"]}, Moves: {g["total_moves"]}, '
      f'Termination: {g["termination"]}, Time: {g["duration_ms"]:.0f}ms')
print('Smoke test passed.')

Running smoke test (depth 3 vs 3)...
  Winner: BLACK, Moves: 20, Termination: total_conversion, Time: 31ms
Smoke test passed.


## Database Setup

Uses a separate `mega_games.sqlite` with two extra columns vs notebook 01:
- `random_first_move` — whether move 1 was randomised
- `batch_label` — tier label for easy filtering (e.g. `"3v6_beginner"`)

In [5]:
os.makedirs('data', exist_ok=True)


def init_mega_db(db_path):
    """Create the mega games and moves tables if they don't exist."""
    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    # WAL mode for concurrent read access during long writes
    c.execute('PRAGMA journal_mode=WAL')

    c.execute('''
        CREATE TABLE IF NOT EXISTS games (
            game_id            INTEGER PRIMARY KEY AUTOINCREMENT,
            white_depth        INTEGER NOT NULL,
            black_depth        INTEGER NOT NULL,
            winner             TEXT    NOT NULL,
            total_moves        INTEGER NOT NULL,
            termination        TEXT    NOT NULL,
            duration_ms        REAL    NOT NULL,
            random_first_move  INTEGER NOT NULL DEFAULT 1,
            batch_label        TEXT    NOT NULL DEFAULT '',
            created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS moves (
            move_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            game_id       INTEGER NOT NULL,
            move_number   INTEGER NOT NULL,
            color         TEXT    NOT NULL,
            from_vertex   TEXT    NOT NULL,
            to_vertex     TEXT    NOT NULL,
            strikes       TEXT,
            upgrades      TEXT,
            board_after   TEXT,
            FOREIGN KEY (game_id) REFERENCES games(game_id)
        )
    ''')

    # Index for checkpoint queries and analysis
    c.execute('''
        CREATE INDEX IF NOT EXISTS idx_games_matchup
        ON games (white_depth, black_depth, random_first_move)
    ''')
    c.execute('''
        CREATE INDEX IF NOT EXISTS idx_moves_game
        ON moves (game_id, move_number)
    ''')

    conn.commit()
    conn.close()
    print(f'Database initialised: {db_path}')


def insert_mega_game(conn, game_record, move_records, random_first_move, batch_label):
    """Insert one game and its moves into the mega database. Returns game_id."""
    c = conn.cursor()
    c.execute(
        '''INSERT INTO games
           (white_depth, black_depth, winner, total_moves, termination,
            duration_ms, random_first_move, batch_label)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
        (
            game_record['white_depth'],
            game_record['black_depth'],
            game_record['winner'],
            game_record['total_moves'],
            game_record['termination'],
            game_record['duration_ms'],
            int(random_first_move),
            batch_label,
        )
    )
    game_id = c.lastrowid

    rows = [
        (game_id, mr['move_number'], mr['color'], mr['from_vertex'],
         mr['to_vertex'], mr['strikes'], mr['upgrades'], mr['board_after'])
        for mr in move_records
    ]
    c.executemany(
        '''INSERT INTO moves
           (game_id, move_number, color, from_vertex, to_vertex,
            strikes, upgrades, board_after)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
        rows,
    )
    return game_id


init_mega_db(DB_PATH)

Database initialised: data/mega_games.sqlite


## Checkpoint / Resume Logic

Count how many games already exist per `(white_depth, black_depth, random_first_move)` tuple
and subtract from the scheduled amount. Safe to re-run after interruptions.

In [6]:
def get_existing_counts(db_path):
    """Return a dict mapping (white_depth, black_depth, random_first_move) -> count."""
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        '''SELECT white_depth, black_depth, random_first_move, COUNT(*)
           FROM games
           GROUP BY white_depth, black_depth, random_first_move'''
    ).fetchall()
    conn.close()
    return {(wd, bd, rfm): cnt for wd, bd, rfm, cnt in rows}


def build_run_plan(schedule, random_fraction, existing_counts):
    """Build a concrete list of sub-batches with remaining game counts.

    Each schedule entry produces TWO sub-batches (random=True and random=False)
    and each sub-batch produces TWO color orderings (a-as-white and b-as-white),
    except when depth_a == depth_b where we only need one ordering.

    Returns a list of dicts ready for the main loop.
    """
    plan = []

    for depth_a, depth_b, n_games, tier in schedule:
        n_random = int(n_games * random_fraction)
        n_deterministic = n_games - n_random

        for rfm, sub_n in [(True, n_random), (False, n_deterministic)]:
            if sub_n <= 0:
                continue

            label = f'{depth_a}v{depth_b}_{tier}'

            if depth_a == depth_b:
                # Symmetric matchup — single ordering is sufficient
                key = (depth_a, depth_b, int(rfm))
                already = existing_counts.get(key, 0)
                remaining = max(0, sub_n - already)
                if remaining > 0:
                    plan.append({
                        'depth_a': depth_a,
                        'depth_b': depth_b,
                        'n_games': remaining,
                        'tier': tier,
                        'label': label,
                        'random_first_move': rfm,
                        'symmetric': True,
                    })
            else:
                # Asymmetric matchup — split evenly into two color orderings:
                #   Ordering 1: WHITE=depth_a, BLACK=depth_b
                #   Ordering 2: WHITE=depth_b, BLACK=depth_a
                half_1 = sub_n // 2
                half_2 = sub_n - half_1

                key_1 = (depth_a, depth_b, int(rfm))
                already_1 = existing_counts.get(key_1, 0)
                remaining_1 = max(0, half_1 - already_1)

                key_2 = (depth_b, depth_a, int(rfm))
                already_2 = existing_counts.get(key_2, 0)
                remaining_2 = max(0, half_2 - already_2)

                if remaining_1 > 0:
                    plan.append({
                        'depth_a': depth_a,
                        'depth_b': depth_b,
                        'n_games': remaining_1,
                        'tier': tier,
                        'label': label,
                        'random_first_move': rfm,
                        'symmetric': False,
                    })
                if remaining_2 > 0:
                    plan.append({
                        'depth_a': depth_b,
                        'depth_b': depth_a,
                        'n_games': remaining_2,
                        'tier': tier,
                        'label': label + '_flip',
                        'random_first_move': rfm,
                        'symmetric': False,
                    })

    return plan


existing = get_existing_counts(DB_PATH)
run_plan = build_run_plan(SCHEDULE, RANDOM_FRACTION, existing)

total_remaining = sum(b['n_games'] for b in run_plan)
total_scheduled = sum(n for _, _, n, _ in SCHEDULE)
total_done = total_scheduled - total_remaining

print(f'Scheduled:  {total_scheduled:>10,} games')
print(f'Already in DB: {total_done:>7,} games')
print(f'Remaining:  {total_remaining:>10,} games')
print(f'Sub-batches to run: {len(run_plan)}')
print()

# Show first few batches
for b in run_plan[:6]:
    rfm_tag = 'random' if b['random_first_move'] else 'determ'
    print(f"  {b['label']:22s}  {b['depth_a']:>2}v{b['depth_b']:<2}  "
          f"{b['n_games']:>7,} games  [{rfm_tag}]")
if len(run_plan) > 6:
    print(f'  ... and {len(run_plan) - 6} more batches')

Scheduled:     500,000 games
Already in DB:       0 games
Remaining:     500,000 games
Sub-batches to run: 66

  3v3_beginner             3v3    18,000 games  [random]
  3v3_beginner             3v3     2,000 games  [determ]
  3v6_beginner             3v6    11,250 games  [random]
  3v6_beginner_flip        6v3    11,250 games  [random]
  3v6_beginner             3v6     1,250 games  [determ]
  3v6_beginner_flip        6v3     1,250 games  [determ]
  ... and 60 more batches


## Calibration (Optional)

Run 10 games per tier to estimate wall-clock time for the full run.
Skip this cell if you just want to go.

In [7]:
CALIBRATION_GAMES = 10

print('Calibrating per-tier speed (10 games each, worst-case matchup)...\n')
calibration = {}

# Pick the SLOWEST (highest combined depth) matchup per tier
tier_samples = {}
for da, db, n, tier in SCHEDULE:
    prev_sum = tier_samples.get(tier, (0, 0, 0))[2]
    if da + db >= prev_sum:
        tier_samples[tier] = (da, db, da + db)
tier_samples = {t: (da, db) for t, (da, db, _) in tier_samples.items()}

for tier, (da, db) in tier_samples.items():
    cfg = AI_CONFIGS[tier]
    t0 = time.perf_counter()
    for i in range(CALIBRATION_GAMES):
        play_one_game(
            da, db, max_moves=MAX_MOVES, seed=i,
            white_ai_kwargs=cfg, black_ai_kwargs=cfg,
            random_first_move=True,
        )
    elapsed = time.perf_counter() - t0
    avg_ms = (elapsed / CALIBRATION_GAMES) * 1000
    calibration[tier] = avg_ms

    tier_total = sum(n for _, _, n, t in SCHEDULE if t == tier)
    est_hours = (avg_ms * tier_total) / (1000 * 3600)
    # Rough parallelism factor
    workers = WORKER_COUNTS[tier]
    est_hours_parallel = est_hours / workers

    print(f'{tier:15s}  {da:>2}v{db:<2}  '
          f'avg {avg_ms:>8,.0f} ms/game  '
          f'{tier_total:>7,} games  '
          f'~{est_hours_parallel:>6.1f} hrs ({workers} workers)')

total_est_hrs = sum(
    (calibration[t] * sum(n for _, _, n, tt in SCHEDULE if tt == t))
    / (1000 * 3600 * WORKER_COUNTS[t])
    for t in calibration
)
print(f'\nEstimated total wall-clock time: ~{total_est_hrs:.1f} hours')

Calibrating per-tier speed (10 games each)...

beginner          3v3   avg      104 ms/game   75,000 games  ~   0.2 hrs (11 workers)
intermediate      3v9   avg       64 ms/game  105,000 games  ~   0.2 hrs (11 workers)
advanced          3v12  avg      102 ms/game  150,000 games  ~   0.4 hrs (11 workers)
expert            9v15  avg      471 ms/game   85,000 games  ~   1.9 hrs (6 workers)
master           12v18  avg      776 ms/game   38,000 games  ~   2.0 hrs (4 workers)
grandmaster      12v20  avg    1,230 ms/game   47,000 games  ~   4.0 hrs (4 workers)

Estimated total wall-clock time: ~8.7 hours


## Main Simulation Loop

In [8]:
# Max futures alive at once — limits peak memory from IPC results
SUBMISSION_CHUNK = 2_000
# Skip board_after JSON in move records (saves ~80% IPC + storage)
SKIP_BOARD_AFTER = True


def run_mega_batch(
    batch,
    db_path,
    ai_configs,
    worker_counts,
    max_moves,
    batch_insert_size,
    deterministic_override,
):
    """Run a single sub-batch with chunked submission and streaming DB writes.

    Key optimizations vs the original version:
    - Futures submitted in chunks (not all at once) to cap peak memory
    - skip_board_after cuts ~80% of IPC payload and storage
    - No result list accumulation — stats tracked via counters
    """
    depth_a = batch['depth_a']
    depth_b = batch['depth_b']
    n_games = batch['n_games']
    tier = batch['tier']
    label = batch['label']
    rfm = batch['random_first_move']

    cfg = dict(ai_configs[tier])
    if not rfm:
        cfg.update(deterministic_override)
    ai_kwargs = dict(cfg)

    num_workers = worker_counts[tier]

    conn = sqlite3.connect(db_path)
    insert_buffer = []
    wins = {}
    total_moves_sum = 0
    games_done = 0

    rfm_tag = 'random' if rfm else 'determ'
    desc = f'{label} [{rfm_tag}]'

    def flush():
        nonlocal insert_buffer
        for gr, mr in insert_buffer:
            insert_mega_game(conn, gr, mr, rfm, label)
        conn.commit()
        insert_buffer = []

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        with tqdm(total=n_games, desc=desc, unit='game', leave=True) as pbar:
            for chunk_start in range(0, n_games, SUBMISSION_CHUNK):
                chunk_end = min(chunk_start + SUBMISSION_CHUNK, n_games)
                futures = {}
                for i in range(chunk_start, chunk_end):
                    task = (depth_a, depth_b, max_moves, i,
                            ai_kwargs, ai_kwargs, rfm, SKIP_BOARD_AFTER)
                    futures[executor.submit(_worker, task)] = True

                for future in as_completed(futures):
                    game_rec, move_recs = future.result()
                    insert_buffer.append((game_rec, move_recs))
                    wins[game_rec['winner']] = wins.get(game_rec['winner'], 0) + 1
                    total_moves_sum += game_rec['total_moves']
                    games_done += 1
                    pbar.update(1)

                    if len(insert_buffer) >= batch_insert_size:
                        flush()

    flush()
    conn.close()

    return {
        'games': games_done,
        'wins': wins,
        'avg_moves': total_moves_sum / games_done if games_done else 0,
    }

In [9]:
# ---- Main execution loop with progress tracking and ETA ----
# No all_results accumulation — summary stats come from the DB afterward.

batch_timings = []
total_games_this_run = 0
run_start = time.perf_counter()

print(f'Starting mega simulation: {total_remaining:,} games across {len(run_plan)} batches')
print(f'Start time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 70)

for batch_idx, batch in enumerate(run_plan):
    batch_start = time.perf_counter()
    rfm_tag = 'random' if batch['random_first_move'] else 'determ'

    print(f'\n[{batch_idx + 1}/{len(run_plan)}] '
          f'{batch["label"]} ({batch["depth_a"]}v{batch["depth_b"]}) '
          f'{batch["n_games"]:,} games [{rfm_tag}] '
          f'— {WORKER_COUNTS[batch["tier"]]} workers')

    stats = run_mega_batch(
        batch,
        DB_PATH,
        AI_CONFIGS,
        WORKER_COUNTS,
        MAX_MOVES,
        BATCH_INSERT_SIZE,
        DETERMINISTIC_OVERRIDE,
    )

    batch_elapsed = time.perf_counter() - batch_start
    total_games_this_run += stats['games']
    batch_timings.append({
        'label': batch['label'],
        'n_games': stats['games'],
        'elapsed_s': batch_elapsed,
        'tier': batch['tier'],
    })

    # Quick stats for this batch
    win_str = ', '.join(f'{k}: {v}' for k, v in sorted(stats['wins'].items()) if v > 0)
    print(f'  Done in {batch_elapsed:.1f}s — {win_str}, avg {stats["avg_moves"]:.1f} moves')

    # ETA based on games completed so far
    total_elapsed = time.perf_counter() - run_start
    games_done = sum(b['n_games'] for b in batch_timings)
    if games_done > 0 and games_done < total_remaining:
        rate = total_elapsed / games_done
        games_left = total_remaining - games_done
        eta_s = rate * games_left
        eta_str = str(timedelta(seconds=int(eta_s)))
        print(f'  Progress: {games_done:,}/{total_remaining:,} '
              f'({games_done/total_remaining*100:.1f}%) — ETA: {eta_str}')

    # Write progress checkpoint
    progress = {
        'last_updated': datetime.now().isoformat(),
        'total_scheduled': total_scheduled,
        'total_remaining_at_start': total_remaining,
        'games_completed_this_run': games_done,
        'batches_completed': batch_idx + 1,
        'batches_total': len(run_plan),
        'elapsed_seconds': total_elapsed,
    }
    with open(PROGRESS_PATH, 'w') as f:
        json.dump(progress, f, indent=2, default=str)

total_elapsed = time.perf_counter() - run_start
print('\n' + '=' * 70)
print(f'Mega simulation complete!')
print(f'Games this run:  {total_games_this_run:,}')
print(f'Total time:      {timedelta(seconds=int(total_elapsed))}')
print(f'Finished at:     {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

Starting mega simulation: 500,000 games across 66 batches
Start time: 2026-03-03 00:52:20

[1/66] 3v3_beginner (3v3) 18,000 games [random] — 11 workers


3v3_beginner [random]:   0%|          | 0/18000 [00:00<?, ?game/s]

  Done in 216.3s — BLACK: 6068, DRAW: 5422, WHITE: 6510, avg 77.4 moves
  Progress: 18,000/500,000 (3.6%) — ETA: 1:36:33

[2/66] 3v3_beginner (3v3) 2,000 games [determ] — 11 workers


3v3_beginner [determ]:   0%|          | 0/2000 [00:00<?, ?game/s]

  Done in 24.6s — BLACK: 669, DRAW: 568, WHITE: 763, avg 73.5 moves
  Progress: 20,000/500,000 (4.0%) — ETA: 1:36:22

[3/66] 3v6_beginner (3v6) 11,250 games [random] — 11 workers


...

  Done in 103.0s — BLACK: 1110, DRAW: 127, WHITE: 13, avg 41.7 moves
  Progress: 43,750/500,000 (8.8%) — ETA: 6:01:08

[6/66] 3v6_beginner_flip (6v3) 1,250 games [determ] — 11 workers


3v6_beginner_flip [determ]:   0%|          | 0/1250 [00:00<?, ?game/s]

  Done in 91.6s — BLACK: 6, DRAW: 128, WHITE: 1116, avg 41.3 moves
  Progress: 45,000/500,000 (9.0%) — ETA: 6:05:34

[7/66] 6v6_beginner (6v6) 27,000 games [random] — 11 workers


6v6_beginner [random]:   0%|          | 0/27000 [00:00<?, ?game/s]

KeyboardInterrupt: 

## Summary Statistics

In [ ]:
# Load full stats from DB (includes any prior runs too)
conn = sqlite3.connect(DB_PATH)

game_count = pd.read_sql('SELECT COUNT(*) AS n FROM games', conn).iloc[0]['n']
move_count = pd.read_sql('SELECT COUNT(*) AS n FROM moves', conn).iloc[0]['n']

print('=' * 60)
print('MEGA DATABASE SUMMARY')
print('=' * 60)
print(f'Total games: {game_count:,}')
print(f'Total moves: {move_count:,}')

db_size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f'Database size: {db_size_mb:.1f} MB')

# Win distribution
wins = pd.read_sql(
    'SELECT winner, COUNT(*) AS games FROM games GROUP BY winner',
    conn,
)
print(f'\nOverall win distribution:')
for _, row in wins.iterrows():
    print(f'  {row["winner"]:6s}: {row["games"]:>8,} ({row["games"]/game_count*100:.1f}%)')

# Termination reasons
terms = pd.read_sql(
    'SELECT termination, COUNT(*) AS games FROM games GROUP BY termination',
    conn,
)
print(f'\nTermination reasons:')
for _, row in terms.iterrows():
    print(f'  {row["termination"]:20s}: {row["games"]:>8,}')

In [ ]:
# Per-tier breakdown
tier_stats = pd.read_sql(
    '''SELECT
           batch_label,
           COUNT(*)                         AS games,
           ROUND(AVG(total_moves), 1)       AS avg_moves,
           ROUND(AVG(duration_ms), 0)       AS avg_ms,
           SUM(CASE WHEN winner='WHITE' THEN 1 ELSE 0 END) AS white_wins,
           SUM(CASE WHEN winner='BLACK' THEN 1 ELSE 0 END) AS black_wins,
           SUM(CASE WHEN winner='DRAW'  THEN 1 ELSE 0 END) AS draws
       FROM games
       GROUP BY batch_label
       ORDER BY batch_label''',
    conn,
)

tier_stats['white_wr'] = (tier_stats['white_wins'] / tier_stats['games'] * 100).round(1)
tier_stats['black_wr'] = (tier_stats['black_wins'] / tier_stats['games'] * 100).round(1)

print('\nPer-batch statistics:')
print(tier_stats.to_string(index=False))

In [ ]:
# Random vs deterministic first-move comparison
rfm_stats = pd.read_sql(
    '''SELECT
           random_first_move,
           COUNT(*)                         AS games,
           ROUND(AVG(total_moves), 1)       AS avg_moves,
           SUM(CASE WHEN winner='WHITE' THEN 1 ELSE 0 END) AS white_wins,
           SUM(CASE WHEN winner='BLACK' THEN 1 ELSE 0 END) AS black_wins,
           SUM(CASE WHEN winner='DRAW'  THEN 1 ELSE 0 END) AS draws
       FROM games
       GROUP BY random_first_move''',
    conn,
)
rfm_stats['random_first_move'] = rfm_stats['random_first_move'].map({1: 'Random', 0: 'Deterministic'})
rfm_stats['white_wr'] = (rfm_stats['white_wins'] / rfm_stats['games'] * 100).round(1)
rfm_stats['black_wr'] = (rfm_stats['black_wins'] / rfm_stats['games'] * 100).round(1)

print('Random vs Deterministic first move:')
print(rfm_stats.to_string(index=False))

conn.close()

## Verify Database

Quick sanity check on the SQLite file.

In [ ]:
conn = sqlite3.connect(DB_PATH)
sample = pd.read_sql(
    '''SELECT g.game_id, g.white_depth, g.black_depth, g.winner,
              g.random_first_move, g.batch_label,
              m.move_number, m.from_vertex, m.to_vertex
       FROM games g
       JOIN moves m ON g.game_id = m.game_id
       WHERE m.move_number <= 3
       ORDER BY g.game_id
       LIMIT 15''',
    conn,
)
conn.close()

print('Sample (first 5 games, first 3 moves each):')
sample

## Run as Script

The standalone `04_mega_simulation.py` has all memory/IPC optimizations
(chunked submission, skip_board_after, no result accumulation).

```bash
# Full run under tmux or nohup:
nohup python3 04_mega_simulation.py > mega_run.log 2>&1 &

# Calibrate first:
python3 04_mega_simulation.py --calibrate

# Check progress:
python3 04_mega_simulation.py --summary
cat data/mega_progress.json | python3 -m json.tool
```

In [ ]:
# Uncomment to auto-export:
# !jupyter nbconvert --to script 04_mega_simulation.ipynb
# print('Exported to 04_mega_simulation.py')